# W05 · 라이브 Berendsen τ_T (덱 8장과 같은 계산)
2D LJ · N 8 · L 7σ · 초기 T 1.5 → 목표 0.6 · τ_T 를 Δt 의 1 · 10 · 100 · 1000 배로

In [ ]:
# ensemble_2d.ipynb 의 lj2d 정의를 그대로 복사
import numpy as np, matplotlib.pyplot as plt
def lj2d(N=8, L=7.0, T0=0.6, Ti=1.5, nvt=True, tau=0.5, steps=4000, dt=0.005, rc=2.5, seed=11):
    rng = np.random.default_rng(seed); dof = 2*N - 2; rc2 = rc*rc; s6c = 1/rc2**3; ecut = 4*(s6c*s6c - s6c)
    nc = int(np.ceil(np.sqrt(N))); sp = min(L/nc, 1.25); off = (L - sp*(nc-1))/2
    pos = np.array([[off + sp*(i % nc), off + sp*(i // nc)] for i in range(N)]) + (rng.random((N, 2)) - 0.5)*0.12
    vel = rng.random((N, 2)) - 0.5; vel -= vel.mean(0); vel *= np.sqrt(Ti*dof/(2*0.5*(vel**2).sum()))
    def forces(p):
        d = p[:, None, :] - p[None, :, :]; d -= L*np.round(d/L); r2 = (d**2).sum(-1); np.fill_diagonal(r2, np.inf)
        m = (r2 < rc2) & (r2 > 0.04); r2m = np.where(m, r2, np.inf); inv = 1/r2m; s6 = inv**3; fr = np.where(m, 24*(2*s6*s6 - s6)*inv, 0)
        return (fr[:, :, None]*d).sum(1)
    f = forces(pos); Ts = []
    for s in range(steps):
        vel += 0.5*dt*f; pos += dt*vel; pos %= L; f = forces(pos); vel += 0.5*dt*f
        T = (vel**2).sum()/dof
        if nvt: vel *= np.clip(np.sqrt(1 + dt/tau*(T0/max(T, 1e-6) - 1)), 0.9, 1.1); T = (vel**2).sum()/dof
        Ts.append(T)
    return np.array(Ts)
exp = np.sqrt(2/14)
fig, ax = plt.subplots(figsize=(9, 4))
for k in (1, 10, 100, 1000):
    Ts = lj2d(tau=k*0.005); tail = Ts[2000:]
    print(f"tau = {k:5d} dt: tail sigma_T/T0 = {tail.std()/0.6:.3f}  (canonical expectation {exp:.3f})")
    ax.plot(Ts, lw=0.8, label=f"tau = {k} dt")
ax.axhline(0.6, color="gray", ls="--"); ax.set_xlabel("step"); ax.set_ylabel("T"); ax.legend(); fig.tight_layout(); plt.show()